# Data cleaning

In [1]:
import pandas as pd
import numpy as np
from configurations import TEST_PATH

"""Load the folded training dataset and the original test dataset."""
train_df = pd.read_csv('train_folds.csv')
test_df = pd.read_csv(TEST_PATH)

print("Missing values in train:")
print(train_df[['Name', 'SexuponOutcome', 'AgeuponOutcome']].isnull().sum())

print("\nMissing values in test:")
print(test_df[['Name', 'SexuponOutcome', 'AgeuponOutcome']].isnull().sum())

Missing values in train:
Name              7691
SexuponOutcome       1
AgeuponOutcome      18
dtype: int64

Missing values in test:
Name              3225
SexuponOutcome       0
AgeuponOutcome       6
dtype: int64


In [2]:
MONTH_CONVERSION = {
    'year': 12.0,
    'month': 1.0,
    'week': 7 / 30.4375,
    'day': 1 / 30.4375
}

def parse_age_in_months(age_str):
    """
    Parse a string representing age and convert it to a continuous 
    numerical value in months.

    Args:
        age_str (str): The age string to parse (e.g., '2 years', '3 weeks').

    Returns:
        float: The converted age in months, or np.nan if parsing fails.
    """
    if pd.isna(age_str):
        return np.nan
    parts = str(age_str).split()
    if len(parts) < 2:
        return np.nan

    number = float(parts[0])
    unit = parts[1].lower().rstrip('s')

    multipliyer = MONTH_CONVERSION.get(unit, np.nan)

    return number * multipliyer

"""Apply the conversion function to create a new 'AgeInMonths' numerical column."""
train_df['AgeInMonths'] = train_df['AgeuponOutcome'].apply(parse_age_in_months)
test_df['AgeInMonths'] = test_df['AgeuponOutcome'].apply(parse_age_in_months)

In [3]:
"""
Impute missing values across the datasets.
Categorical columns ('Name', 'SexuponOutcome') are filled with constant placeholders.
The numerical column ('AgeInMonths') is imputed using the median age of the corresponding 'AnimalType'.

To prevent data leakage, the training set imputation is performed fold-by-fold 
using out-of-fold medians. The test set is imputed using overall training set medians.
"""
for df in [train_df, test_df]:
    df['Name'] = df['Name'].fillna('No Name')
    df['SexuponOutcome'] = df['SexuponOutcome'].fillna('Unknown')

for fold in range(5):
    train_fold = train_df[train_df['fold'] != fold]
    val_idx = train_df[train_df['fold'] == fold].index

    type_medians = train_fold.groupby('AnimalType')['AgeInMonths'].median()

    for animal_type, median_val in type_medians.items():
        mask = (train_df.index.isin(val_idx)) & (train_df['AnimalType'] == animal_type) & (train_df['AgeInMonths'].isnull())
        train_df.loc[mask, 'AgeInMonths'] = median_val

overall_medians = train_df.groupby('AnimalType')['AgeInMonths'].median()
for animal_type, median_val in overall_medians.items():
    test_df.loc[(test_df['AnimalType'] == animal_type) & (test_df['AgeInMonths'].isnull()), 'AgeInMonths'] = median_val

In [4]:
"""
Verify that no missing values remain in the specified columns,
then save the cleaned datasets for the next pipeline steps.
"""
print("Remaining missing values in train_df:")
print(train_df[['Name', 'SexuponOutcome', 'AgeInMonths']].isnull().sum())

print("\nRemaining missing values in test_df:")
print(test_df[['Name', 'SexuponOutcome', 'AgeInMonths']].isnull().sum())

train_df.to_csv('train_cleaned.csv', index=False)
test_df.to_csv('test_cleaned.csv', index=False)
print("\ntrain_cleaned.csv' and 'test_cleaned.csv' saved successfully!")

Remaining missing values in train_df:
Name              0
SexuponOutcome    0
AgeInMonths       0
dtype: int64

Remaining missing values in test_df:
Name              0
SexuponOutcome    0
AgeInMonths       0
dtype: int64

train_cleaned.csv' and 'test_cleaned.csv' saved successfully!
